# Credit Risk Score - ML Project
## 1. Problem & Setup
Using **MLFlow** to track the model performance.

Using **Polars** instead of **Pandas** for data management.

Using dataset from [Kaggle](https://www.kaggle.com/datasets/laotse/credit-risk-dataset).

In [107]:
DATASET_PATH = "D:\\Codes\\ML Credit Risk\\credit_risk_dataset.csv"

In [108]:
import polars as pl

df = pl.read_csv('credit_risk_dataset.csv')

In [109]:
df.head()

person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
i64,i64,str,f64,str,str,i64,f64,i64,f64,str,i64
22,59000,"""RENT""",123.0,"""PERSONAL""","""D""",35000,16.02,1,0.59,"""Y""",3
21,9600,"""OWN""",5.0,"""EDUCATION""","""B""",1000,11.14,0,0.1,"""N""",2
25,9600,"""MORTGAGE""",1.0,"""MEDICAL""","""C""",5500,12.87,1,0.57,"""N""",3
23,65500,"""RENT""",4.0,"""MEDICAL""","""C""",35000,15.23,1,0.53,"""N""",2
24,54400,"""RENT""",8.0,"""MEDICAL""","""C""",35000,14.27,1,0.55,"""Y""",4


In [110]:
print("Schema (column names and data types):")
for col, dtype in df.schema.items():
    print(f"{col}: {dtype}")

Schema (column names and data types):
person_age: Int64
person_income: Int64
person_home_ownership: String
person_emp_length: Float64
loan_intent: String
loan_grade: String
loan_amnt: Int64
loan_int_rate: Float64
loan_status: Int64
loan_percent_income: Float64
cb_person_default_on_file: String
cb_person_cred_hist_length: Int64


In [111]:
print(df['person_home_ownership'].unique())

shape: (4,)
Series: 'person_home_ownership' [str]
[
	"RENT"
	"OTHER"
	"MORTGAGE"
	"OWN"
]


In [112]:
df = df.with_columns(
    pl.col("person_home_ownership").cast(pl.Categorical)
);

C:\Users\karti\AppData\Local\Temp\ipykernel_15952\1189026022.py:1: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  df = df.with_columns(


In [113]:
print(df['loan_intent'].unique())

shape: (6,)
Series: 'loan_intent' [str]
[
	"DEBTCONSOLIDATION"
	"PERSONAL"
	"VENTURE"
	"MEDICAL"
	"EDUCATION"
	"HOMEIMPROVEMENT"
]


In [114]:
df = df.with_columns(
    pl.col("loan_intent").cast(pl.Categorical)
);

C:\Users\karti\AppData\Local\Temp\ipykernel_15952\347706905.py:1: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  df = df.with_columns(


In [115]:
print(df['loan_grade'].unique())

shape: (7,)
Series: 'loan_grade' [str]
[
	"E"
	"A"
	"F"
	"D"
	"G"
	"C"
	"B"
]


In [116]:
grade_order = ["A", "B", "C", "D", "E", "F", "G"]

df = df.with_columns(
    pl.col("loan_grade")
    .cast(pl.Enum(grade_order))
)

In [117]:
print(df['loan_status'].value_counts())

shape: (2, 2)
┌─────────────┬───────┐
│ loan_status ┆ count │
│ ---         ┆ ---   │
│ i64         ┆ u32   │
╞═════════════╪═══════╡
│ 0           ┆ 25473 │
│ 1           ┆ 7108  │
└─────────────┴───────┘


In [118]:
print(df['loan_percent_income'].head())

shape: (10,)
Series: 'loan_percent_income' [f64]
[
	0.59
	0.1
	0.57
	0.53
	0.55
	0.25
	0.45
	0.44
	0.42
	0.16
]


In [119]:
print(df['cb_person_default_on_file'].head())

shape: (10,)
Series: 'cb_person_default_on_file' [str]
[
	"Y"
	"N"
	"N"
	"N"
	"Y"
	"N"
	"N"
	"N"
	"N"
	"N"
]


In [120]:
df = df.with_columns(
    pl.when(pl.col("cb_person_default_on_file") == "Y")
    .then(1)
    .otherwise(0)
    .alias("cb_person_default_on_file")
)

In [121]:
print(df['cb_person_cred_hist_length'].head())

shape: (10,)
Series: 'cb_person_cred_hist_length' [i64]
[
	3
	2
	3
	2
	4
	2
	3
	4
	2
	3
]


Let's review the columns and take notes.
- **person_age:** The age of the person, in integer format.
- **person_income:** The income of the person, in integer format as well.
- **person_home_ownership:** The type of ownership of a home, previously in String format, transformed to Category.
- **person_emp_length:** Employment length, in years. Stored as integer.
- **loan_intent:** The intention of the person to take the loan. Stored as string, turned into categorical.
- **loan_grade:** The grade of the loan. The order matters here, so it is transformed into a category with that in consideration.
- **loan_amnt:** The amount of loan received. Stored as integer.
- **loan_int_rate:** Interest rate. Stored as float.
- **loan_status:** The target variable. 1 is default (risky borrower) and 0 is non-default (safe borrower). The `value_counts` output shows that the dataset is *moderately imbalanced*, but not extremely (78/22 split). It may be a good idea to rely on metrics *other than Accuracy*. Class weights may be set as balanced during model training.
- **loan_percent_income:** The rate of loan amount to annual income. This is a strong predictor of default risk.
- **cb_person_default_on_file:** "Y" values mean that the person has had previous default history, and "N" values mean that the person has not had previous default history. This was mapped to binary.
- **cb_person_cred_hist_length:** The length of the credit history. It is stored as integer.

In [122]:
df = df.unique()

In [123]:

print("\nNull counts per column:")

df.null_count()


Null counts per column:


person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,887,0,0,0,3095,0,0,0,0


In [124]:
print("\nDescriptive statistics:")
df.describe()


Descriptive statistics:


statistic,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
str,f64,f64,str,f64,str,str,f64,f64,f64,f64,f64,f64
"""count""",32416.0,32416.0,"""32416""",31529.0,"""32416""","""32416""",32416.0,29321.0,32416.0,32416.0,32416.0,32416.0
"""null_count""",0.0,0.0,"""0""",887.0,"""0""","""0""",0.0,3095.0,0.0,0.0,0.0,0.0
"""mean""",27.747008,66091.640826,null,4.79051,null,null,9593.845632,11.017265,0.218688,0.17025,0.176765,5.811297
"""std""",6.3541,62015.580269,null,4.14549,null,null,6322.730241,3.24168,0.413363,0.106812,0.381475,4.05903
"""min""",20.0,4000.0,null,0.0,null,null,500.0,5.42,0.0,0.0,0.0,2.0
"""25%""",23.0,38556.0,null,2.0,null,null,5000.0,7.9,0.0,0.09,0.0,3.0
"""50%""",26.0,55000.0,null,4.0,null,null,8000.0,10.99,0.0,0.15,0.0,4.0
"""75%""",30.0,79200.0,null,7.0,null,null,12250.0,13.47,0.0,0.23,0.0,8.0
"""max""",144.0,6e6,null,123.0,null,null,35000.0,23.22,1.0,0.83,1.0,30.0


### Notes
**Columns with missing values:**
- person_emp_length
- loan_int_rate

We may use SimpleImputer later for these.

**Age outlier:**
- 144 years old as max.

We may use clip method for this.

**Skewed income:**
- Mean is 66k yet the max is quite high.
- Heavy right skew, strong outliers.

We may apply logarithmic transform.

In [125]:
# Handle missing values in person_emp_length and loan_int_rate
df = df.with_columns(
    pl.col('person_emp_length').fill_null(pl.col('person_emp_length').mean()),
    pl.col('loan_int_rate').fill_null(pl.col('loan_int_rate').mean())
)

# Clip age outlier (assuming max reasonable age is 100)
df = df.with_columns(pl.col('person_age').clip(0, 100))

# Apply logarithmic transform to person_income to handle skewness
df = df.with_columns(pl.col('person_income').log().alias('person_income_log'))

In [126]:
print("\nDescriptive statistics:")
df.describe()


Descriptive statistics:


statistic,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_income_log
str,f64,f64,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64
"""count""",32416.0,32416.0,"""32416""",32416.0,"""32416""","""32416""",32416.0,32416.0,32416.0,32416.0,32416.0,32416.0,32416.0
"""null_count""",0.0,0.0,"""0""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",27.741517,66091.640826,null,4.79051,null,null,9593.845632,11.017265,0.218688,0.17025,0.176765,5.811297,10.925635
"""std""",6.274489,62015.580269,null,4.088378,null,null,6322.730241,3.08304,0.413363,0.106812,0.381475,4.05903,0.565819
"""min""",20.0,4000.0,null,0.0,null,null,500.0,5.42,0.0,0.0,0.0,2.0,8.29405
"""25%""",23.0,38556.0,null,2.0,null,null,5000.0,8.49,0.0,0.09,0.0,3.0,10.559867
"""50%""",26.0,55000.0,null,4.0,null,null,8000.0,11.017265,0.0,0.15,0.0,4.0,10.915088
"""75%""",30.0,79200.0,null,7.0,null,null,12250.0,13.11,0.0,0.23,0.0,8.0,11.279732
"""max""",100.0,6e6,null,123.0,null,null,35000.0,23.22,1.0,0.83,1.0,30.0,15.60727


In [127]:
print("Schema (column names and data types):")
for col, dtype in df.schema.items():
    print(f"{col}: {dtype}")

Schema (column names and data types):
person_age: Int64
person_income: Int64
person_home_ownership: Categorical(ordering='physical')
person_emp_length: Float64
loan_intent: Categorical(ordering='physical')
loan_grade: Enum(categories=['A', 'B', 'C', 'D', 'E', 'F', 'G'])
loan_amnt: Int64
loan_int_rate: Float64
loan_status: Int64
loan_percent_income: Float64
cb_person_default_on_file: Int32
cb_person_cred_hist_length: Int64
person_income_log: Float64


## Model Training
We can now build the pipelines and begin training and testing our model from this point on.

In [128]:
num_features = [
    "person_age",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "person_income_log"
]

cat_features = [
    "person_home_ownership",
    "loan_intent"
]

ordinal_features = [
    "loan_grade"
]

binary_features = [
    "cb_person_default_on_file"
]

target = "loan_status"

In [129]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("ord", OrdinalEncoder(categories=[["A","B","C","D","E","F","G"]]), ordinal_features),
        ("bin", "passthrough", binary_features)
    ]
)

In [130]:
df = df.to_pandas()
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_income_log
0,23,140000,MORTGAGE,7.0,PERSONAL,A,12000,8.49,0,0.09,0,3,11.849398
1,23,39996,RENT,2.0,EDUCATION,B,18000,9.25,1,0.45,0,4,10.596535
2,22,54000,MORTGAGE,1.0,HOMEIMPROVEMENT,D,10000,14.61,1,0.19,1,4,10.896739
3,31,53922,RENT,15.0,HOMEIMPROVEMENT,B,5000,10.59,0,0.09,0,5,10.895294
4,31,42100,RENT,4.0,DEBTCONSOLIDATION,B,5000,10.36,0,0.12,0,9,10.647803


In [131]:
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

target = "loan_status"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        class_weight="balanced",
        random_state=42
    ),
    "gradient_boosting": GradientBoostingClassifier(),
    "xgboost": XGBClassifier(
        n_estimators=200,
        use_label_encoder=False,
        eval_metric="auc",
        random_state=42
    ),
    "catboost": CatBoostClassifier(
        iterations=200,
        verbose=0,
        random_state=42
    )
}

mlflow.set_experiment("credit-risk-models")

best_model_name = None
best_auc = -1
best_pipeline = None

for name, model in models.items():

    with mlflow.start_run(run_name=name):

        pipeline = Pipeline([
            ("preprocess", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

        auc = roc_auc_score(y_test, y_pred_proba)

        mlflow.log_param("model_name", name)
        mlflow.log_metric("roc_auc", auc)

        mlflow.sklearn.log_model(pipeline, "model")

        print(f"{name} AUC: {auc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_model_name = name
            best_pipeline = pipeline


print("\nBEST MODEL:")
print(best_model_name, best_auc)

with mlflow.start_run(run_name="best_model_final"):
    mlflow.log_param("best_model", best_model_name)
    mlflow.log_metric("best_roc_auc", best_auc)
    mlflow.sklearn.log_model(best_pipeline, "model")


c:\Users\karti\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/11 12:07:30 INFO mlflow.tracking.fluent: Experiment with name 'credit-risk-models' does not exist. Creating a new experiment.
2026/05/11 12:07:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 12:07:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommend

logistic_regression AUC: 0.8700


2026/05/11 12:08:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 12:08:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


random_forest AUC: 0.9360


2026/05/11 12:08:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 12:08:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

c:\Users\karti\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:08:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/05/11 12:08:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 12:08:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


xgboost AUC: 0.9482


2026/05/11 12:09:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/11 12:09:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/11 12:09:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


catboost AUC: 0.9466

BEST MODEL:
xgboost 0.9481565479534739


2026/05/11 12:09:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [132]:
import pickle
import os

# Save the best model locally
model_dir = "models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, f"best_model_{best_model_name}.pkl")
with open(model_path, "wb") as f:
    pickle.dump(best_pipeline, f)

print(f"Best model saved to {model_path}")
print(f"Model: {best_model_name}")
print(f"AUC Score: {best_auc:.4f}")

Best model saved to models\best_model_xgboost.pkl
Model: xgboost
AUC Score: 0.9482
